# For Google Colab 

**Uncomment when you run this file on Google Colab**

In [1]:
'''
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

!mkdir '/content/drive/MyDrive/finalTerm'
!mkdir '/content/drive/MyDrive/finalTerm/dataset'

!pip install kaggle

from google.colab import files
# kaggle.json : Kaggle API token : Personal key
    # You can download your kaggle.json file from Kaggle website
    # 1. Enter https://www.kaggle.com/settings/account
    # 2. API > 'Create New Token'
    # 3. Download kaggle.json file
# Upload your kaggle.json file
files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Dataset download
%cd /content/drive/MyDrive/finalTerm/dataset
!kaggle competitions download -c postech25-csed-441-final-project

# Unzip the dataset
!unzip postech25-csed-441-final-project.zip
'''

"\nfrom google.colab import drive\n\ndrive.mount('/content/drive', force_remount=True)\n\n!mkdir '/content/drive/MyDrive/finalTerm'\n!mkdir '/content/drive/MyDrive/finalTerm/dataset'\n\n!pip install kaggle\n\nfrom google.colab import files\n# kaggle.json : Kaggle API token : Personal key\n    # You can download your kaggle.json file from Kaggle website\n    # 1. Enter https://www.kaggle.com/settings/account\n    # 2. API > 'Create New Token'\n    # 3. Download kaggle.json file\n# Upload your kaggle.json file\nfiles.upload()\n\n!mkdir -p ~/.kaggle\n!mv kaggle.json ~/.kaggle/\n!chmod 600 ~/.kaggle/kaggle.json\n\n# Dataset download\n%cd /content/drive/MyDrive/finalTerm/dataset\n!kaggle competitions download -c postech25-csed-441-final-project\n\n# Unzip the dataset\n!unzip postech25-csed-441-final-project.zip\n"

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Dataset & DataLoader
This section is baseline.

You can change this section if you want.

In [3]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
import PIL
import os

class TrainDataset(Dataset):
    def __init__(self, root_path, transform, data_aug=None):
        super(TrainDataset,self).__init__()
        self.augmentation = data_aug
        self.root_path = root_path
        self.transform = transform
        self.image_list = list()
        self.annotation = pd.read_csv(os.path.join(self.root_path, '../Train_64.csv'))

        self.image_list = np.array(self.annotation.values.tolist())[:, 0]
        self.labels = np.array(self.annotation.values.tolist())[:, 1]

    def __len__(self):
        return len(self.annotation)

    def __getitem__(self,index):
        img = PIL.Image.open(os.path.join(self.root_path,str('Train_64'),self.image_list[index]))
        if self.augmentation is not None:
            img = self.augmentation(img)
        img = self.transform(img)
        return img, int(self.labels[index])

class TestDataset(Dataset):
    def __init__(self, root_path, transform, data_aug=None):
        super(TestDataset,self).__init__()
        self.root_path = root_path
        self.transform = transform
        self.image_list = list()
        self.annotation = pd.read_csv(os.path.join(self.root_path, '../Test_64.csv'))

        self.image_list = np.array(self.annotation.values.tolist())[:, 0]


    def __len__(self):
        return len(self.annotation)
    
    def __getitem__(self,index):

        img = PIL.Image.open(os.path.join(self.root_path,str('Test_64'),self.image_list[index]))
        img = self.transform(img)
        return img

In [4]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# --- [수정됨] 15 Epoch용 적당한 Augmentation ---
train_transform = transforms.Compose([
    transforms.RandomCrop(64, padding=4),       # 위치 변화 학습 (필수)
    transforms.RandomHorizontalFlip(),          # 좌우 반전 (필수, 학습 효율 최고)
    # Rotation, ColorJitter 제거 -> 학습 속도 향상 및 과소적합 방지
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
# (나머지 DataLoader 코드는 그대로 사용)

# 경로 설정 (Kaggle 환경 기준, 로컬/Colab이면 경로 수정 필요)
# 예: root_path = '/content/drive/MyDrive/finalTerm/dataset/Train_64/'
TRAIN_PATH = '../input/postech25-csed-441-final-project/Train_64/'
TEST_PATH = '../input/postech25-csed-441-final-project/Test_64/'

# 로컬 테스트용 경로 (필요시 주석 해제하여 사용)
# TRAIN_PATH = './dataset/Train_64/' # 예시
# TEST_PATH = './dataset/Test_64/'   # 예시

train_dataset = TrainDataset(root_path=TRAIN_PATH, transform=train_transform)
test_dataset = TestDataset(root_path=TEST_PATH, transform=test_transform)

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=64, # 메모리 상황에 따라 조절 (64~256)
                          shuffle=True,
                          num_workers=2,
                          drop_last=True)

test_loader = DataLoader(dataset=test_dataset,
                         batch_size=64,
                         shuffle=False,
                         num_workers=2)

print("Data Loaders Ready!")

Data Loaders Ready!


# Your Awesome Model

In [5]:
import torch.nn as nn
import torch.nn.functional as F

class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.view(b, c, -1).mean(dim=2)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

class BottleneckBlock(nn.Module):
    expansion = 4 

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        
        self.se = SEBlock(out_channels * self.expansion)
        self.act = nn.GELU() # ReLU 대신 GELU 사용
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.act(self.bn1(self.conv1(x)))
        out = self.act(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = self.se(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        return self.act(out)

class SENet(nn.Module):
    def __init__(self, num_classes=15):
        super().__init__()
        self.in_channels = 64
        
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU()
        )

        # Layers: [blocks, channels, stride]
        self.layer1 = self._make_layer(64, 3, stride=1)
        self.layer2 = self._make_layer(128, 4, stride=2)
        self.layer3 = self._make_layer(256, 6, stride=2) # Deep Layer
        self.layer4 = self._make_layer(512, 3, stride=2)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3) # 과적합 방지
        self.fc = nn.Linear(512 * BottleneckBlock.expansion, num_classes)
        
        self._init_weights()

    def _make_layer(self, out_channels, blocks, stride):
        downsample = None
        if stride != 1 or self.in_channels != out_channels * BottleneckBlock.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels * BottleneckBlock.expansion, 
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * BottleneckBlock.expansion),
            )

        layers = []
        layers.append(BottleneckBlock(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels * BottleneckBlock.expansion
        
        for _ in range(1, blocks):
            layers.append(BottleneckBlock(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gap(x)
        x = x.flatten(1)
        x = self.dropout(x)
        x = self.fc(x)
        return x
device = torch.device('cuda')
model = SENet(num_classes=15)
model = model.to(device)

# 파라미터 수 확인
num_params = sum(p.numel() for p in model.parameters())
print(f'Total Parameters: {num_params:,}')

Total Parameters: 26,046,031


# Model parameter checking

In [6]:
# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

The number of your model parameters : 26046031
Parameter usage : 26.046031%


# Model training

In [7]:
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR # 핵심 변경 사항!
import tqdm

# --- [설정 변경] ---
EPOCHS = 15  # 제한된 에폭
MAX_LR = 0.005 # OneCycleLR은 학습률을 평소보다 높게 잡아야 효과가 좋습니다.

# 1. Loss (Label Smoothing 유지)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# 2. Optimizer (AdamW 유지)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# 3. Scheduler (OneCycleLR 적용)
# steps_per_epoch는 train_loader의 길이여야 합니다.
scheduler = OneCycleLR(optimizer, 
                       max_lr=MAX_LR, 
                       epochs=EPOCHS, 
                       steps_per_epoch=len(train_loader),
                       pct_start=0.3) # 전체 학습의 30% 시점까지 LR 상승

print(f"Training Start... (Epochs: {EPOCHS} with OneCycleLR)")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        scheduler.step() # OneCycleLR은 배치마다 step()을 호출합니다! (중요)
        
        train_loss += loss.item()
        _, predicted = output.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()
        
        pbar.set_postfix({'Loss': loss.item(), 'Acc': 100.*correct/total, 'LR': scheduler.get_last_lr()[0]})
    
    avg_loss = train_loss / len(train_loader)
    avg_acc = 100. * correct / total
    
    print(f"Epoch: {epoch+1} | Avg Loss: {avg_loss:.4f} | Avg Acc: {avg_acc:.2f}%")

Training Start... (Epochs: 15 with OneCycleLR)


Epoch 1/15: 100%|██████████| 703/703 [04:59<00:00,  2.35it/s, Loss=1.99, Acc=34.2, LR=0.000762]


Epoch: 1 | Avg Loss: 2.1765 | Avg Acc: 34.18%


Epoch 2/15: 100%|██████████| 703/703 [04:50<00:00,  2.42it/s, Loss=1.79, Acc=45.6, LR=0.00218] 


Epoch: 2 | Avg Loss: 1.9004 | Avg Acc: 45.61%


...

Epoch: 13 | Avg Loss: 0.9921 | Avg Acc: 83.22%


Epoch 14/15: 100%|██████████| 703/703 [04:48<00:00,  2.44it/s, Loss=0.967, Acc=85.8, LR=0.000111]


Epoch: 14 | Avg Loss: 0.9352 | Avg Acc: 85.77%


Epoch 15/15: 100%|██████████| 703/703 [04:48<00:00,  2.44it/s, Loss=0.894, Acc=87.1, LR=2.02e-8]

Epoch: 15 | Avg Loss: 0.9063 | Avg Acc: 87.07%


# Submit
Do not edit the submission code below.

In [8]:
submit = pd.read_csv('../input/postech25-csed-441-final-project/Test_64.csv')

# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

total_prediction = list()
model.eval()
with torch.no_grad():
    for x in tqdm.tqdm(test_loader):
        x = torch.FloatTensor(x).cuda()
        output = model(x)
        predict = torch.argmax(output,dim=1)
        total_prediction.extend(predict.cpu().numpy())
    submit['label'] = total_prediction
    submit.to_csv('submission.csv',index=False)

The number of your model parameters : 26046031
Parameter usage : 26.046031%


100%|██████████| 118/118 [00:22<00:00,  5.20it/s]


# Submit in Google Colab
**Uncomment when you run this file on Google Colab**

In [9]:
'''
# !kaggle competitions submit -c postech25-csed-441-final-project -f submission.csv -m "<Your commit message>"
!kaggle competitions submit -c postech25-csed-441-final-project -f submission.csv -m "MY Submission"
'''

'\n# !kaggle competitions submit -c postech25-csed-441-final-project -f submission.csv -m "<Your commit message>"\n!kaggle competitions submit -c postech25-csed-441-final-project -f submission.csv -m "MY Submission"\n'